# Task 23 — Training & Deploying TensorFlow Models at Scale
## Phases 1–6 | Practical Notebook

**Goal:** Export and inspect a TensorFlow SavedModel, serve it locally, test model versioning and batching concepts, convert it for edge deployment, and explore GPU configuration and execution.

**Important execution notes**
- Run cells from top to bottom. Some sections require Docker, `saved_model_cli`, gRPC packages, an NVIDIA GPU, or a cloud account.
- The notebook includes checks and clearly labels optional/hardware-dependent experiments. If a required service is unavailable, record that limitation instead of inventing results.
- Phase 4 uses a **design-document approach** by default to avoid unexpected cloud charges. Do not upload credentials or service-account keys to GitHub.
- This notebook uses MNIST as the primary lightweight example, matching the README.

## 0. Environment setup

Install the core packages in your active environment. If you are using Google Colab, TensorFlow and many dependencies may already be installed.

For Docker serving, install Docker Desktop separately and ensure it is running. For gRPC, install the optional packages in the next cell.

In [ ]:
# If needed, uncomment and run in a notebook environment:
# %pip install -U tensorflow numpy matplotlib requests
# %pip install tensorflow-serving-api grpcio

In [ ]:
import os
import sys
import json
import time
import math
import platform
import statistics
import subprocess
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("TensorFlow:", tf.__version__)
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))

# Phase 1 — Exporting & Inspecting SavedModels

A SavedModel is a portable TensorFlow representation containing a serialized computation graph, variables, and optional assets. The version number is represented as a directory name so multiple versions can coexist.

### 1.1 Load and prepare MNIST

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

# Keep the first run manageable; use the full dataset if you have time/resources.
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

### 1.2 Train or build a small baseline classifier

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(28, 28), name="image"),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax", name="class_probabilities"),
], name="mnist_classifier")

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=128,
    verbose=1,
)
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_accuracy:.4f}")

### 1.3 Export a versioned SavedModel

In [ ]:
MODEL_NAME = "my_mnist_model"
VERSION_1 = "0001"
MODEL_ROOT = Path(MODEL_NAME)
MODEL_PATH_1 = MODEL_ROOT / VERSION_1
MODEL_ROOT.mkdir(exist_ok=True)

# Keras 3 uses model.export() for a TensorFlow SavedModel.
# tf.saved_model.save() also writes a SavedModel for supported Keras models.
if MODEL_PATH_1.exists():
    import shutil
    shutil.rmtree(MODEL_PATH_1)

tf.saved_model.save(model, str(MODEL_PATH_1))
print("Exported to:", MODEL_PATH_1.resolve())
print("Top-level contents:", sorted(p.name for p in MODEL_PATH_1.iterdir()))

### 1.4 Inspect the SavedModel directory

In [ ]:
for root, dirs, files in os.walk(MODEL_PATH_1):
    level = Path(root).relative_to(MODEL_PATH_1).parts
    indent = "  " * len(level)
    print(f"{indent}{Path(root).name}/")
    for filename in sorted(files):
        print(f"{indent}  {filename}")

Expected structure includes `saved_model.pb`, a `variables/` directory, and possibly `assets/`. Exact contents vary by TensorFlow/Keras version.

- `saved_model.pb`: serialized model graph and signatures.
- `variables/`: stored model weights.
- `assets/`: optional external assets such as vocabulary files.

### 1.5 Reload and compare predictions

In [ ]:
X_sample = X_test[:5]
keras_predictions = model.predict(X_sample, verbose=0)

loaded_saved_model = tf.saved_model.load(str(MODEL_PATH_1))
print("Available signatures:", list(loaded_saved_model.signatures.keys()))

# The default serving signature is generally the most portable call interface.
serving_fn = loaded_saved_model.signatures["serving_default"]
print("Input signature:", serving_fn.structured_input_signature)
print("Output signature:", serving_fn.structured_outputs)

# Exported Keras SavedModels can expose a named input such as 'image'.
# Use the signature's actual input name rather than assuming it.
signature_inputs = serving_fn.structured_input_signature[1]
input_name = next(iter(signature_inputs))
saved_outputs = serving_fn(**{input_name: tf.constant(X_sample, dtype=tf.float32)})
output_name = next(iter(saved_outputs))
saved_predictions = saved_outputs[output_name].numpy()

print("Output key:", output_name)
print("Keras output shape:", keras_predictions.shape)
print("SavedModel output shape:", saved_predictions.shape)
print("Max absolute prediction difference:",
      float(np.max(np.abs(keras_predictions - saved_predictions))))
assert np.allclose(keras_predictions, saved_predictions, atol=1e-5),     "SavedModel predictions differ from original model."
print("Self-test passed.")

### 1.6 Inspect using `saved_model_cli` (optional)

In [ ]:
# Run this in a terminal if the command is installed:
# saved_model_cli show --dir my_mnist_model/0001 --all
#
# The CLI may not be installed or exposed on PATH in every environment.

### 1.7 Bundle preprocessing inside the model

In [ ]:
# This version accepts raw pixel values in [0, 255] and rescales internally.
raw_input_model = keras.Sequential([
    keras.layers.Input(shape=(28, 28), dtype=tf.float32, name="raw_image"),
    keras.layers.Rescaling(1.0 / 255.0),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
], name="mnist_with_preprocessing")

raw_input_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
raw_input_model.fit(
    (X_train[:20000] * 255.0), y_train[:20000],
    validation_split=0.1,
    epochs=2,
    batch_size=128,
    verbose=1,
)

RAW_MODEL_PATH = MODEL_ROOT / "raw_preprocessing" / "0001"
RAW_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
if RAW_MODEL_PATH.exists():
    import shutil
    shutil.rmtree(RAW_MODEL_PATH)
tf.saved_model.save(raw_input_model, str(RAW_MODEL_PATH))

raw_sample = (X_test[:5] * 255.0).astype("float32")
raw_fn = tf.saved_model.load(str(RAW_MODEL_PATH)).signatures["serving_default"]
raw_input_name = next(iter(raw_fn.structured_input_signature[1]))
raw_output = next(iter(raw_fn(**{raw_input_name: tf.constant(raw_sample)}).values())).numpy()
print("Raw-input output shape:", raw_output.shape)

**Why bundle preprocessing?** It reduces the chance that training and serving apply different preprocessing. Clients can send raw inputs without separately reproducing scaling.

**SavedModel limitation:** exportability depends on supported TensorFlow operations and the model's serving signature. Python-only callbacks such as `tf.py_function` may not be portable to a serving environment.

### 1.8 Export helper with post-export self-test

In [ ]:
def export_model(model_to_export, name, version, test_batch):
    """Export a model to name/version and verify the SavedModel predictions."""
    path = Path(name) / str(version)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        import shutil
        shutil.rmtree(path)
    tf.saved_model.save(model_to_export, str(path))

    loaded = tf.saved_model.load(str(path))
    fn = loaded.signatures["serving_default"]
    input_key = next(iter(fn.structured_input_signature[1]))
    actual_outputs = fn(**{input_key: tf.constant(test_batch, dtype=tf.float32)})
    actual = next(iter(actual_outputs.values())).numpy()
    expected = model_to_export(test_batch, training=False).numpy()
    max_diff = float(np.max(np.abs(actual - expected)))
    assert np.allclose(actual, expected, atol=1e-5), f"Self-test failed: {max_diff}"
    return {"path": str(path), "max_abs_difference": max_diff, "passed": True}

export_result = export_model(model, "export_helper_demo", "0001", X_test[:5])
export_result

# Phase 2 — TensorFlow Serving with Docker: REST vs. gRPC

TensorFlow Serving provides a dedicated model-serving process. The README's typical Docker setup exposes **8501 for REST** and **8500 for gRPC**.

Prerequisites: Docker must be installed and running, and the exported model directory must be accessible to the container.

### 2.1 Check Docker availability

In [ ]:
try:
    result = subprocess.run(
        ["docker", "--version"], capture_output=True, text=True, timeout=10
    )
    print(result.stdout.strip() or result.stderr.strip())
except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
    print("Docker CLI unavailable or timed out:", exc)
    print("Install/start Docker Desktop, or document this phase as not run.")

### 2.2 Start TensorFlow Serving (run in a terminal)

In [ ]:
# Example for a POSIX shell:
# docker pull tensorflow/serving
# docker run --rm -p 8500:8500 -p 8501:8501 \
#   -v "$(pwd)/my_mnist_model:/models/my_mnist_model" \
#   -e MODEL_NAME=my_mnist_model tensorflow/serving
#
# Windows PowerShell: replace $(pwd) with an absolute Windows path in the
# Docker volume argument, e.g. C:/path/to/project/my_mnist_model:/models/my_mnist_model

### 2.3 Query REST endpoint (run only while TF Serving is running)

In [ ]:
import requests

REST_URL = "http://localhost:8501/v1/models/my_mnist_model:predict"

def predict_rest(batch, url=REST_URL, timeout=10):
    payload = {"signature_name": "serving_default", "instances": batch.tolist()}
    response = requests.post(url, json=payload, timeout=timeout)
    response.raise_for_status()
    data = response.json()
    if "predictions" not in data:
        raise RuntimeError(f"Unexpected TF Serving response: {data}")
    return np.asarray(data["predictions"])

# Uncomment after starting the server:
# rest_predictions = predict_rest(X_test[:5])
# print("Max REST difference:", np.max(np.abs(rest_predictions - keras_predictions)))

### 2.4 Measure JSON payload overhead

In [ ]:
batch = X_test[:100].astype(np.float32)
raw_float32_bytes = batch.nbytes
json_payload = json.dumps({"instances": batch.tolist()}).encode("utf-8")
json_bytes = len(json_payload)

print(f"Raw float32 payload: {raw_float32_bytes:,} bytes")
print(f"JSON request body:   {json_bytes:,} bytes")
print(f"JSON/raw ratio:      {json_bytes / raw_float32_bytes:.2f}x")

### 2.5 gRPC client (optional; requires server and packages)

In [ ]:
# Uncomment after installing tensorflow-serving-api and grpcio and starting TF Serving.
#
# import grpc
# from tensorflow_serving.apis import predict_pb2, prediction_service_pb2_grpc
# from tensorflow_serving.apis import get_model_status_pb2
# from tensorflow_serving.apis import model_pb2
# from tensorflow.python.framework import tensor_util
#
# def predict_grpc(batch, model_name="my_mnist_model"):
#     channel = grpc.insecure_channel("localhost:8500")
#     stub = prediction_service_pb2_grpc.PredictionServiceStub(channel)
#     request = predict_pb2.PredictRequest()
#     request.model_spec.name = model_name
#     request.model_spec.signature_name = "serving_default"
#     request.inputs["image"].CopyFrom(tf.make_tensor_proto(batch, dtype=tf.float32))
#     response = stub.Predict(request, timeout=10.0)
#     output_name = next(iter(response.outputs))
#     return tf.make_ndarray(response.outputs[output_name])
#
# Note: the exact input/output signature names depend on your export.
# Use the printed signature from Phase 1 rather than assuming "image".

### 2.6 Benchmarking template

In [ ]:
def benchmark_callable(fn, repeats=5, warmup=1):
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        fn()
        times.append((time.perf_counter() - start) * 1000)
    return {
        "mean_ms": float(np.mean(times)),
        "std_ms": float(np.std(times, ddof=1)) if len(times) > 1 else 0.0,
        "min_ms": float(np.min(times)),
        "max_ms": float(np.max(times)),
    }

# Local-model benchmark example; replace with predict_rest/predict_grpc once
# the serving service is running to compare protocols.
local_benchmark = benchmark_callable(lambda: model.predict(X_test[:32], verbose=0))
local_benchmark

**Protocol rule of thumb from the README:** REST is convenient and widely compatible for smaller requests; gRPC is often preferable for large tensors and high-throughput clients. Measure both in your own environment before drawing conclusions.

For production, do not expose an insecure gRPC channel publicly; configure TLS and appropriate authentication.

# Phase 3 — Versioning, Rollback, Batching & Scaling Out

This phase demonstrates the version-directory pattern locally. Automatic model reload, traffic routing and batching require a running TensorFlow Serving process and should be verified in its logs.

### 3.1 Export model version 0002

In [ ]:
MODEL_PATH_2 = MODEL_ROOT / "0002"
model_v2 = keras.Sequential([
    keras.layers.Input(shape=(28, 28), name="image"),
    keras.layers.Flatten(),
    keras.layers.Dense(96, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
], name="mnist_classifier_v2")
model_v2.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_v2.fit(X_train[:30000], y_train[:30000], epochs=2, batch_size=128, verbose=1)

if MODEL_PATH_2.exists():
    import shutil
    shutil.rmtree(MODEL_PATH_2)
tf.saved_model.save(model_v2, str(MODEL_PATH_2))
print("Versions present:", sorted(p.name for p in MODEL_ROOT.iterdir() if p.is_dir()))

### 3.2 Compare predictions from both versions

In [ ]:
pred_v1 = model.predict(X_test[:10], verbose=0)
pred_v2 = model_v2.predict(X_test[:10], verbose=0)
print("V1 predicted labels:", pred_v1.argmax(axis=1))
print("V2 predicted labels:", pred_v2.argmax(axis=1))
print("Predictions identical:", np.array_equal(pred_v1.argmax(axis=1), pred_v2.argmax(axis=1)))

### 3.3 Rollout and rollback procedure

1. Export the new model into a new version directory (for example `0002`).
2. Check TF Serving logs and model-status endpoints to verify the new version loaded.
3. Send a small share of traffic to the new version, if version-specific routing is configured.
4. Monitor latency, error rate and a model-quality metric.
5. Roll back by restoring the previous serving configuration or removing the new version directory, then verify the active version.

**Do not claim a live rollout or rollback unless you actually run it against the serving process.** Deleting a version directory is not a substitute for verifying serving state and logs.

### 3.4 Batching and scale-out notes

TensorFlow Serving supports automatic request batching when configured with the relevant server flags and batching parameters. Batching can improve throughput, but waiting to form a batch may increase latency.

For scale-out, a typical architecture is:

`Clients → Load balancer → Multiple TF Serving instances → Monitoring`

A real experiment should compare batching off/on and report throughput plus p50, p95 and p99 latency under the same workload. Multi-instance serving also needs health checks, resource sizing and rolling-update procedures.

# Phase 4 — A Managed Prediction Service in the Cloud

**Default approach: design document only.** The README allows a design-document deliverable when you do not have cloud access or prefer not to incur charges. The original instructions reference Google's older AI Platform; current Google Cloud managed ML deployment uses Vertex AI, so confirm current product settings and commands before a hands-on deployment.

### 4.1 Deployment plan

1. Decide the model, region, runtime and expected request volume.
2. Create a cloud project and configure billing only if proceeding hands-on.
3. Set a budget alert and use least-privilege service accounts.
4. Upload the versioned SavedModel to private cloud storage.
5. Create the managed model and endpoint; configure minimum/maximum replicas.
6. Authenticate using the platform's recommended identity mechanism, not a personal credential file embedded in code.
7. Test predictions against local outputs and record latency/errors.
8. Add monitoring and alerts for latency, failures and cost.
9. Delete endpoints, model resources, storage artifacts and temporary credentials when finished.

### 4.2 Cold-start decision table

In [ ]:
cloud_decisions = [
    {"Scenario": "Hobby project", "Choice": "Scale to zero / low minimum", "Trade-off": "Lower idle cost; cold starts possible"},
    {"Scenario": "Startup with spiky traffic", "Choice": "Autoscaling with sensible limits", "Trade-off": "Balance responsiveness and cost"},
    {"Scenario": "Latency-critical service", "Choice": "Keep minimum capacity warm", "Trade-off": "Higher ongoing cost; fewer cold starts"},
]
import pandas as pd
pd.DataFrame(cloud_decisions)

### 4.3 Security checklist

- [ ] Keep model buckets private unless public access is explicitly required.
- [ ] Grant only the permissions required by the serving identity.
- [ ] Never commit service-account keys, tokens or credentials to version control.
- [ ] Use TLS, authentication, request validation and rate limiting.
- [ ] Set budget alerts before creating paid resources.
- [ ] Record resource deletion and final cost (or a clearly labelled estimate).

# Phase 5 — Mobile, Embedded & Browser Deployment

TensorFlow Lite is designed for efficient on-device inference. Quantization can reduce model size and, depending on the operators and runtime, improve latency and energy efficiency. TensorFlow.js supports running compatible models in JavaScript environments such as browsers.

### 5.1 Convert SavedModel to TFLite

In [ ]:
TFLITE_PATH = Path("mnist_float32.tflite")
converter = tf.lite.TFLiteConverter.from_saved_model(str(MODEL_PATH_1))
tflite_model = converter.convert()
TFLITE_PATH.write_bytes(tflite_model)
print("Saved:", TFLITE_PATH, "| bytes:", TFLITE_PATH.stat().st_size)

### 5.2 Run inference with the TFLite interpreter

In [ ]:
def tflite_predict(model_bytes, batch):
    interpreter = tf.lite.Interpreter(model_content=model_bytes)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    results = []
    for item in batch:
        interpreter.set_tensor(input_details[0]["index"], item[None, ...].astype(input_details[0]["dtype"]))
        interpreter.invoke()
        results.append(interpreter.get_tensor(output_details[0]["index"])[0])
    return np.asarray(results)

tflite_predictions = tflite_predict(tflite_model, X_test[:20])
original_predictions = model.predict(X_test[:20], verbose=0)
print("TFLite shape:", tflite_predictions.shape)
print("Max absolute difference:", float(np.max(np.abs(tflite_predictions - original_predictions))))
print("Model size (bytes):", len(tflite_model))

### 5.3 Float16 quantization

In [ ]:
converter_f16 = tf.lite.TFLiteConverter.from_saved_model(str(MODEL_PATH_1))
converter_f16.optimizations = [tf.lite.Optimize.DEFAULT]
converter_f16.target_spec.supported_types = [tf.float16]
tflite_f16 = converter_f16.convert()
Path("mnist_float16.tflite").write_bytes(tflite_f16)

f16_predictions = tflite_predict(tflite_f16, X_test[:100])
baseline_predictions = model.predict(X_test[:100], verbose=0)
print("Float16 file bytes:", len(tflite_f16))
print("Max probability difference:", float(np.max(np.abs(f16_predictions - baseline_predictions))))
print("Baseline accuracy on sample:", float(np.mean(baseline_predictions.argmax(1) == y_test[:100])))
print("Float16 accuracy on sample:", float(np.mean(f16_predictions.argmax(1) == y_test[:100])))

### 5.4 Symmetric 8-bit quantization arithmetic

In [ ]:
weights = np.array([-1.5, -0.4, 0.0, 0.3, 0.8], dtype=np.float32)
m = float(np.max(np.abs(weights)))
scale = m / 127.0
q = np.clip(np.round(weights / scale), -127, 127).astype(np.int8)
dequantized = q.astype(np.float32) * scale
max_abs_error = float(np.max(np.abs(weights - dequantized)))

print("Weights:      ", weights)
print("Quantized:    ", q)
print("Dequantized:  ", dequantized)
print("Scale:        ", scale)
print("Max abs error:", max_abs_error)

### 5.5 Post-training dynamic-range quantization

In [ ]:
converter_int8_weights = tf.lite.TFLiteConverter.from_saved_model(str(MODEL_PATH_1))
converter_int8_weights.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_dynamic = converter_int8_weights.convert()
Path("mnist_dynamic_quantized.tflite").write_bytes(tflite_dynamic)

dynamic_predictions = tflite_predict(tflite_dynamic, X_test[:100])
print("Dynamic quantized size:", len(tflite_dynamic))
print("Accuracy on first 100:", float(np.mean(dynamic_predictions.argmax(1) == y_test[:100])))
print("Max probability difference:", float(np.max(np.abs(dynamic_predictions - model.predict(X_test[:100], verbose=0)))))

### 5.6 Full-integer quantization with representative data

In [ ]:
def representative_dataset():
    for i in range(200):
        yield [X_train[i:i+1].astype(np.float32)]

converter_full_int = tf.lite.TFLiteConverter.from_saved_model(str(MODEL_PATH_1))
converter_full_int.optimizations = [tf.lite.Optimize.DEFAULT]
converter_full_int.representative_dataset = representative_dataset
converter_full_int.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_full_int.inference_input_type = tf.int8
converter_full_int.inference_output_type = tf.int8

try:
    tflite_full_int = converter_full_int.convert()
    Path("mnist_full_integer.tflite").write_bytes(tflite_full_int)
    print("Full-integer conversion succeeded; size:", len(tflite_full_int))
    print("Inspect input/output quantization parameters before inference.")
except Exception as exc:
    tflite_full_int = None
    print("Full-integer conversion failed in this TensorFlow version:", repr(exc))

### 5.7 Model comparison table

In [ ]:
def evaluate_tflite_bytes(label, model_bytes, x, y):
    preds = tflite_predict(model_bytes, x)
    return {
        "Model": label,
        "Size (bytes)": len(model_bytes),
        "Accuracy (sample)": float(np.mean(preds.argmax(axis=1) == y)),
    }

comparison_rows = [
    {
        "Model": "SavedModel (directory)",
        "Size (bytes)": sum(p.stat().st_size for p in MODEL_PATH_1.rglob("*") if p.is_file()),
        "Accuracy (sample)": float(np.mean(model.predict(X_test[:500], verbose=0).argmax(1) == y_test[:500])),
    },
    evaluate_tflite_bytes("TFLite float32", tflite_model, X_test[:500], y_test[:500]),
    evaluate_tflite_bytes("TFLite float16", tflite_f16, X_test[:500], y_test[:500]),
    evaluate_tflite_bytes("TFLite dynamic range", tflite_dynamic, X_test[:500], y_test[:500]),
]
pd.DataFrame(comparison_rows)

### 5.8 TensorFlow.js

For browser deployment, the converter typically generates a `model.json` file plus binary weight shards. The browser loads these files and runs inference locally. This can help with offline access, latency and privacy, but model size, browser memory and update strategy must be considered.

Conversion commands depend on the installed TensorFlow.js converter version; use the current TensorFlow.js documentation before running `tensorflowjs_converter`.

# Phase 6 — Using GPUs: Memory, Placement & Parallel Execution

This phase begins with device discovery. Most CPU-only environments can run the conceptual exercises; GPU-specific demonstrations will be skipped or reported as unavailable when no compatible GPU is present.

### 6.1 Inspect GPU availability

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
print("Physical GPUs:", gpus)

try:
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=10)
    print(smi.stdout if smi.returncode == 0 else smi.stderr)
except (FileNotFoundError, subprocess.TimeoutExpired) as exc:
    print("nvidia-smi unavailable:", exc)

### 6.2 Configure memory growth (must be done before GPU initialization)

In [ ]:
# Run this cell early, before creating GPU tensors or models, if desired.
# If TensorFlow has already initialized a GPU, this may raise RuntimeError.
for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
        print("Enabled memory growth for", gpu)
    except RuntimeError as exc:
        print("Could not change memory growth; GPU may already be initialized:", exc)

### 6.3 Device placement and soft placement

In [ ]:
tf.debugging.set_log_device_placement(False)
tf.config.set_soft_device_placement(True)

with tf.device("/CPU:0"):
    cpu_tensor = tf.constant([1.0, 2.0, 3.0])
print("CPU tensor device:", cpu_tensor.device)

with tf.device("/GPU:0"):
    try:
        gpu_tensor = tf.constant([1.0, 2.0, 3.0])
        print("Requested GPU tensor device:", gpu_tensor.device)
    except Exception as exc:
        print("GPU placement unavailable:", repr(exc))

int_tensor = tf.constant([1, 2, 3], dtype=tf.int32)
print("Integer tensor device:", int_tensor.device)

### 6.4 Compare simple CPU and GPU execution (if GPU is available)

In [ ]:
def matrix_multiply_on(device, size=1024, repeats=5):
    a = tf.random.normal((size, size))
    b = tf.random.normal((size, size))
    # Warm up
    with tf.device(device):
        _ = tf.matmul(a, b).numpy()
    times = []
    with tf.device(device):
        for _ in range(repeats):
            start = time.perf_counter()
            _ = tf.matmul(a, b).numpy()  # .numpy() synchronizes execution
            times.append((time.perf_counter() - start) * 1000)
    return {"device": device, "mean_ms": float(np.mean(times)),
            "std_ms": float(np.std(times, ddof=1)) if repeats > 1 else 0.0}

results = [matrix_multiply_on("/CPU:0")]
if tf.config.list_physical_devices("GPU"):
    results.append(matrix_multiply_on("/GPU:0"))
pd.DataFrame(results)

### 6.5 Thread-pool settings (CPU experiment)

In [ ]:
# TensorFlow may already have initialized its runtime; settings can only be changed
# before runtime initialization. If this raises RuntimeError, restart the kernel
# and set the values near the beginning of the notebook.
print("Logical CPU devices:", tf.config.list_logical_devices("CPU"))
print("This environment's default TensorFlow threading is retained for this run.")

### 6.6 Parallel-execution concept

TensorFlow executes operations when their input dependencies are ready. CPU execution can use inter-op parallelism (different operations at once) and intra-op parallelism (splitting one operation across threads). GPU kernels often perform substantial internal parallel work themselves.

Practical rules:
- Keep CPU preprocessing efficient and feed the accelerator with prepared batches.
- Minimize unnecessary CPU↔GPU transfers.
- Keep large datasets in CPU memory when they are not needed on the GPU at that moment.
- Measure before assuming more GPUs or more threads will improve performance.

### 6.7 Optional virtual GPU experiment

In [ ]:
# Virtual GPUs can only be configured before TensorFlow initializes the physical GPU.
# Restart the kernel first, then run this cell before any GPU operations.
#
# physical_gpus = tf.config.list_physical_devices("GPU")
# if physical_gpus:
#     tf.config.set_logical_device_configuration(
#         physical_gpus[0],
#         [tf.config.LogicalDeviceConfiguration(memory_limit=2048),
#          tf.config.LogicalDeviceConfiguration(memory_limit=2048)]
#     )
#     print(tf.config.list_logical_devices("GPU"))
# else:
#     print("No physical GPU available; virtual GPU demonstration skipped.")
#
# Note: this config splits a GPU into logical devices for testing; it does not
# provide the performance of two physical GPUs.

## Final status checklist — Phases 1–6

Mark each item only after you've run it or completed the documented alternative.

- [ ] Phase 1: SavedModel exported, inspected and self-tested.
- [ ] Phase 2: REST client tested; gRPC client and protocol benchmark tested if Docker is available.
- [ ] Phase 3: Version 2 exported and local predictions compared; serving rollout/rollback verified only if TF Serving was run.
- [ ] Phase 4: Cloud design, cost choices and security checklist documented (or cloud deployment completed and resources removed).
- [ ] Phase 5: TFLite conversions tested; size and accuracy results recorded.
- [ ] Phase 6: Device inventory captured; placement and CPU/GPU timing experiments run where supported.
- [ ] No credentials or private keys committed to the repository.

**Important:** The notebook is a practical scaffold. Record actual outputs from your own environment; do not replace missing results with guessed measurements.